In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.model_selection import ShuffleSplit
import os
from tqdm.auto import tqdm

# ==========================================
# 1. METRICS, HELPERS & CLUSTERING
# ==========================================
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    if denom == 0.0: return 0.0
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

class LgbTqdmCallback:
    def __init__(self, max_iter, desc="Boosting"):
        self.max_iter = max_iter
        self.pbar = tqdm(total=max_iter, desc=desc, leave=False)
    def __call__(self, env):
        self.pbar.update(1)
        if env.evaluation_result_list:
            score = env.evaluation_result_list[0][2]
            self.pbar.set_postfix({'val_rmse': f'{score:.5f}'})
    def __del__(self):
        self.pbar.close()

def get_predictive_clusters(X, y, codes, n_clusters=5):
    base_model = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.1, n_jobs=-1, random_state=42, verbosity=-1)
    base_model.fit(X, y)
    preds = base_model.predict(X)
    
    profile_df = pd.DataFrame({'code': codes, 'actual': y, 'pred': preds, 'error': preds - y})
    stock_profiles = profile_df.groupby('code').agg(
        mean_actual=('actual', 'mean'), mean_pred=('pred', 'mean'), mean_error=('error', 'mean')
    ).fillna(0) 
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
    stock_profiles['cluster'] = kmeans.fit_predict(stock_profiles)
    return stock_profiles['cluster'].to_dict()

# ==========================================
# 2. DATA PREP
# ==========================================
print("Loading data...")
base_path = "./" 
train = pd.read_parquet(os.path.join(base_path, "train.parquet"))
test = pd.read_parquet(os.path.join(base_path, "test.parquet"))

cat_cols = ['code', 'sub_code', 'sub_category']
feature_cols = [c for c in train.columns if c.startswith('feature_')]

for col in cat_cols:
    le = LabelEncoder()
    full_data = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(full_data)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

params = {
    'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
    'learning_rate': 0.05, 'num_leaves': 31, 'feature_fraction': 0.8,
    'verbosity': -1, 'seed': 42, 'n_jobs': -1
}
N_ESTIMATORS = 800
N_CLUSTERS = 5 
horizons = train['horizon'].unique()

# ==========================================
# RUN 1: 10x REPEATED SPLITS (85% / 15%)
# ==========================================
print("\n" + "="*50)
print(" RUN 1: 10x SHUFFLE SPLIT (85/15)")
print("="*50)

# ShuffleSplit gives us exactly 10 iterations of 85/15 splits.
rs = ShuffleSplit(n_splits=10, test_size=0.15, random_state=42)
horizon_cv_scores = {h: [] for h in horizons}

for h in tqdm(horizons, desc="Horizons (10x CV)"):
    train_h = train[train['horizon'] == h].copy()
    
    fold = 1
    for train_idx, val_idx in rs.split(train_h):
        tr_df = train_h.iloc[train_idx].copy()
        val_df = train_h.iloc[val_idx].copy()
        
        cluster_map = get_predictive_clusters(X=tr_df[feature_cols], y=tr_df['y_target'], codes=tr_df['code'], n_clusters=N_CLUSTERS)
        tr_df['cluster'] = tr_df['code'].map(cluster_map).fillna(0).astype(int)
        val_df['cluster'] = val_df['code'].map(cluster_map).fillna(0).astype(int)
        val_df['val_preds'] = 0.0
        
        for c_id in range(N_CLUSTERS):
            c_tr = tr_df[tr_df['cluster'] == c_id]
            c_val = val_df[val_df['cluster'] == c_id]
            if len(c_tr) == 0: continue
                
            model = lgb.LGBMRegressor(**params, n_estimators=N_ESTIMATORS)
            # Only doing early stopping internally, muting the inner progress bar to avoid console spam across 10 folds
            if len(c_val) > 0:
                model.fit(
                    c_tr[feature_cols], c_tr['y_target'], sample_weight=c_tr['weight'],
                    eval_set=[(c_val[feature_cols], c_val['y_target'])], eval_sample_weight=[c_val['weight']],
                    callbacks=[lgb.early_stopping(50, verbose=False)]
                )
                val_df.loc[val_df['cluster'] == c_id, 'val_preds'] = model.predict(c_val[feature_cols])
                
        score = weighted_rmse_score(val_df['y_target'].values, val_df['val_preds'].values, val_df['weight'].values)
        horizon_cv_scores[h].append(score)
        tqdm.write(f"  [Horizon {h} - Fold {fold}] Score: {score:.5f}")
        fold += 1
        
    avg_h_score = np.mean(horizon_cv_scores[h])
    tqdm.write(f"✓ Horizon {h} AVERAGE CV Score: {avg_h_score:.5f}\n")

# ==========================================
# RUN 2: FULL DATASET & SELF-ACCURACY CHECK
# ==========================================
print("\n" + "="*50)
print(" RUN 2: FULL TRAINING PIPELINE (100% DATA)")
print("="*50)

test['y_target'] = 0.0
train['full_train_preds'] = 0.0 # Container for full dataset self-predictions

for h in tqdm(horizons, desc="Horizons (Full Training)"):
    train_h = train[train['horizon'] == h].copy()
    test_h = test[test['horizon'] == h].copy()
    
    tqdm.write(f"  [Horizon {h}] Clustering 100% of stocks...")
    cluster_map = get_predictive_clusters(X=train_h[feature_cols], y=train_h['y_target'], codes=train_h['code'], n_clusters=N_CLUSTERS)
    
    train_h['cluster'] = train_h['code'].map(cluster_map).fillna(0).astype(int)
    test_h['cluster'] = test_h['code'].map(cluster_map).fillna(0).astype(int)
    
    # We must determine a fixed n_estimators. If you overfit here, this is the culprit.
    FIXED_TREES = 300 
    
    for c_id in range(N_CLUSTERS):
        c_tr = train_h[train_h['cluster'] == c_id]
        c_te = test_h[test_h['cluster'] == c_id]
        if len(c_tr) == 0: continue
            
        model = lgb.LGBMRegressor(**params, n_estimators=FIXED_TREES)
        tqdm_cb = LgbTqdmCallback(max_iter=FIXED_TREES, desc=f"Cluster {c_id}")
        
        model.fit(c_tr[feature_cols], c_tr['y_target'], sample_weight=c_tr['weight'], callbacks=[tqdm_cb])
        tqdm_cb.pbar.close()
        
        # 1. Self-Predict (Training Accuracy)
        train.loc[c_tr.index, 'full_train_preds'] = model.predict(c_tr[feature_cols])
        
        # 2. Final Test Predict
        if len(c_te) > 0:
            test.loc[c_te.index, 'y_target'] = model.predict(c_te[feature_cols])

# Calculate exact self-accuracy on the full training dataset
self_accuracy_score = weighted_rmse_score(train['y_target'].values, train['full_train_preds'].values, train['weight'].values)
print(f"\n⚠ FULL TRAINING SET SELF-ACCURACY SCORE: {self_accuracy_score:.5f}")
print("If this number is radically lower than your 10-fold CV average, you are memorizing the training set.")

submission = test[['id', 'y_target']].copy()
submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' generated!")

Loading data...

 RUN 1: 10x SHUFFLE SPLIT (85/15)


Horizons (10x CV):   0%|          | 0/4 [01:12<?, ?it/s]

  [Horizon 25 - Fold 1] Score: 0.46093


Horizons (10x CV):   0%|          | 0/4 [02:04<?, ?it/s]

  [Horizon 25 - Fold 2] Score: 0.44649


...

  [Horizon 10 - Fold 4] Score: 0.37569


Horizons (10x CV):  75%|███████▌  | 3/4 [5:21:02<1:14:12, 4452.28s/it]

  [Horizon 10 - Fold 5] Score: 0.37650


Horizons (10x CV):  75%|███████▌  | 3/4 [5:21:08<1:47:02, 6422.84s/it]


KeyboardInterrupt: 